In [1]:
import torch
import monai
import pandas as pd
import os

from tqdm.auto import tqdm
from torch import nn
from monai.networks.nets import ResNet
from torch.amp import GradScaler, autocast

device = torch.device("cuda" if torch.cuda.is_available else "cpu")
device

device(type='cuda')

In [12]:
path_img = "data/niftis_utCGpHE/"
path_file = "data/train_labels_JNDlMjr.csv"

df = pd.read_csv(path_file)

df['image'] = df['uid'].apply(lambda x: os.path.abspath(os.path.join(path_img, f"{x}.nii.gz")))
df['label'] = df['is_pathologic'].astype(int)

dict_df = []
miss = 0

for _, row in df.iterrows():
    if os.path.exists(row['image']):
        dict_df.append({
            'image': row['image'],
            'label': row['label']
        })
    else:
        miss += 1


transforms = monai.transforms.Compose([
    monai.transforms.LoadImaged(keys = ['image']),
    monai.transforms.EnsureChannelFirstd(keys = ['image']),
    monai.transforms.Orientationd(keys = ['image'], axcodes = 'RAS'),
    monai.transforms.Spacingd(keys = ['image'], pixdim = (2.0 ,2.0, 2.0), mode = 'bilinear'),
    monai.transforms.ResizeWithPadOrCropd(keys = ['image'], spatial_size = (64, 64, 64)),
    monai.transforms.ScaleIntensityd(keys = ['image'], minv = 0.0, maxv = 1.0),
    monai.transforms.NormalizeIntensityd(keys = ['image'], nonzero = True)
])

img_data = monai.data.PersistentDataset(
    data = dict_df,
    transform = transforms,
    cache_dir = "~/dataset/monai_cache"
)

dataloader = monai.data.DataLoader(dataset = img_data, batch_size = 1, shuffle = True, num_workers = 4)


dict_df[:6]

/home/abhit/neuron/GOD1/lib/python3.14/site-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)
/home/abhit/neuron/GOD1/lib/python3.14/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  s

[{'image': '/home/abhit/neuron/data/niftis_utCGpHE/xaji0y6d.nii.gz',
  'label': 0},
 {'image': '/home/abhit/neuron/data/niftis_utCGpHE/pbhsahxt.nii.gz',
  'label': 0},
 {'image': '/home/abhit/neuron/data/niftis_utCGpHE/hv3a3zmf.nii.gz',
  'label': 0},
 {'image': '/home/abhit/neuron/data/niftis_utCGpHE/8mdd4v30.nii.gz',
  'label': 0},
 {'image': '/home/abhit/neuron/data/niftis_utCGpHE/t9nt3w5u.nii.gz',
  'label': 1},
 {'image': '/home/abhit/neuron/data/niftis_utCGpHE/zbikcidk.nii.gz',
  'label': 1}]

In [13]:
model = monai.networks.nets.resnet10(
    spatial_dims = 3,
    n_input_channels = 1,
    num_classes = 1,
)

lossfn = monai.losses.FocalLoss()
optimizer = torch.optim.AdamW(params = model.parameters(), lr = 1e-4, weight_decay = 1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer = optimizer, T_max = 50, eta_min = 1e-6)


In [15]:
NUM_EPOCH = 3

scaler = GradScaler()

def train(dataloader: monai.data.dataloader, optimizer: torch.optim.Optimizer, lossfn: torch.nn.Module):

    model.train()

    train_loss = 0.0

    for batch in dataloader:

        image = batch['image']
        labels = batch['label']

        labels = labels.long().unsqueeze(dim=1)

        optimizer.zero_grad()

        with autocast(device.type):
            train = model(image)
            loss = lossfn(train, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    train_loss /= len(dataloader)

    return train_loss

def run():

    for epoch in tqdm(range(NUM_EPOCH)):
        train_loss = train(
            dataloader = dataloader,
            optimizer = optimizer,
            lossfn = lossfn
        )

        scheduler.step()

        print(f"Epoch: {epoch}| train: {train_loss}")
 


In [ ]:
run()

  0%|          | 0/3 [00:00<?, ?it/s]

/home/abhit/neuron/GOD1/lib/python3.14/site-packages/torch/utils/data/dataloader.py:430: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [13]:
import torch

torch.save(model.state_dict(), "parkinson.pth")

In [10]:
model.load_state_dict(torch.load("parkinson.pth"))
model.eval()

data = "data/niftis_utCGpHE/w3zp08j3.nii.gz"
#sample = {'image':data}

datas = transforms(sample)

datas = datas['image'].unsqueeze(0)

with torch.no_grad():

    out = model(datas)

print(out.item())

0.3095259368419647


In [9]:
sample = {'image':data}
sample

{'image': 'data/niftis_utCGpHE/w3zp08j3.nii.gz'}